# YouTube Engagement Analysis
EDA and like-prediction models on US YouTube trending data. Run all in Colab; upload the two files from `data/` when prompted.


In [ ]:
import os
if not (os.path.exists('/content/2-US_youtube_trending_data.csv.gz') and os.path.exists('/content/1-US_category_id.json')):
    from google.colab import files
    print('Upload 2-US_youtube_trending_data.csv.gz and 1-US_category_id.json (in the repo root)')
    files.upload()


In [ ]:
!pip -q install squarify
!mkdir -p /content/ytdata
!gunzip -c /content/2-US_youtube_trending_data.csv.gz > /content/ytdata/US_youtube_trending_data.csv
!cp /content/1-US_category_id.json /content/ytdata/US_category_id.json


In [ ]:
#importing required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
import ipywidgets as widgets


In [ ]:
# ignoring warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# loading data into the dataframe
data = pd.read_csv("/content/ytdata/US_youtube_trending_data.csv")


In [ ]:
# loading random samples to understand data
data.sample(n = 5)

In [ ]:
# Loading another dataframe 
df = pd.read_json("/content/ytdata/US_category_id.json")

In [ ]:
# loading random samples
df.sample(n = 3)

In [ ]:
# Evaluating Null Values
data.isnull().sum()

In [ ]:
# Plotting Null values in a heatmap
plt.figure(figsize=(5,5))
sns.heatmap(data.isnull())
plt.title("Null Values Heatmap")

In [ ]:
# dropping non-required columns
data.drop(columns=['description','tags','thumbnail_link'],inplace=True)

In [ ]:
# Rechecking Null Values
data.isnull().sum()

In [ ]:
# General Info of data
data.info()

Converting datatype of date columns and creating more columns for further analysis 

In [ ]:
data['publishedAt'] = pd.to_datetime(data['publishedAt'])

In [ ]:
data['trending_date'] = pd.to_datetime(data['trending_date'])

In [ ]:
data.info()

In [ ]:
data['p_year'] = data['publishedAt'].dt.year

In [ ]:
data['t_year'] = data['trending_date'].dt.year

In [ ]:
data['p_month'] = data['publishedAt'].dt.month_name()

In [ ]:
data['t_month'] = data['trending_date'].dt.month_name()

In [ ]:
data

In [ ]:
# Shape of data
data.shape

In [ ]:
data['video_id'].nunique()

In [ ]:
len(data[data['video_id'] == '3C66w5Z0ixs'])

There are only 36279 unique video_id in the whole 1,99,190 records. So it implies that the video_id are getting repeated. As a solution I'm grouping the data with video_id 

In [ ]:
video = data.groupby('video_id').max().reset_index()

In [ ]:
video

In [ ]:
# Rows & Columns of grouped dataframe 
video.shape

Extracting ID's from second dataframe(df) to create a new dataframe which would contain Id's and categories

In [ ]:
# Extracting Id's
ids = [df['items'][i]['id'] for i in range(32) ]

In [ ]:
df['items'][2]['snippet']['title']

In [ ]:
#Extracting categories
gen = [df['items'][i]['snippet']['title'] for i in range(32) ]

In [ ]:
gen

In [ ]:
# creating new DataFrame
genre = pd.DataFrame({'categoryId' : ids , 'Genre' : gen})


In [ ]:
genre.sample(n = 3)

In [ ]:
genre.info()

In [ ]:
# Changing the datatype of categoryId from Object to Int
genre['categoryId'] = genre['categoryId'].astype(int)

Merging two Dataframes video & genre

In [ ]:
video.head(1)

In [ ]:
genre.head(1)

In [ ]:
video = pd.merge(video,genre,on = 'categoryId',how = 'inner')

# Exploratory Data Analysis

In [ ]:
video

### No. Of videos released Annually

In [ ]:
annually_published = video.groupby(by = 'p_year').count()

In [ ]:
annually_published

In [ ]:
# Plotting Annual released videos
plt.figure(figsize=(5,5))
sns.barplot(x = annually_published.index , y = annually_published['video_id'])
plt.title("Video's Released Each Year.")
plt.xlabel("Years")
plt.ylabel("no. of videos published")

### Distribution of Genre's

In [ ]:
# Getting the distribution of each Genre
category_counts = video['Genre'].value_counts()

In [ ]:

# Create a figure for Genre distribution
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot()

# Create a treemap using squarify library
squarify.plot(sizes=category_counts.values, 
              label=category_counts.index, 
              alpha=0.8, ax=ax)

# Add title and display plot
plt.title("Treemap of Video Genre's")
plt.show()


### Plotting Trending Genre's each Year

In [ ]:

plt.figure(figsize=(15,8))
sns.countplot(x='t_year', hue='Genre', data=video)
plt.title('Trending Categories Each Year')
plt.xlabel('Year')
plt.ylabel('Number of Videos')
plt.show()


### Visualizing Genre's w.r.t Numerical Features using Ipywidgets
(view_count,likes,dislikes,comment_count)

In [ ]:
conn = widgets.Dropdown(
    options=[ 'view_count','likes', 'dislikes','comment_count'],
    value='likes',
    description='Select option:',
    disabled=False,
)


In [ ]:
conn

In [ ]:
def graph(a):
  plt.figure(figsize=(9,9))
  sns.barplot(y = 'Genre', x = a , data = video, estimator = sum)
  plt.show()


In [ ]:
widgets.interact(graph,a = conn)

### Getting Top 5 Videos based On Genre's & Numerical Features using Ipywidgets

In [ ]:

c = widgets.Dropdown(
    options=gen,
    value='Music',
    description='Number:',
    disabled=False,
)

conn = widgets.Dropdown(
    options=['view_count', 'likes', 'dislikes', 'comment_count'],
    value='likes',
    description='Select option:',
    disabled=False,
)

def graph(a,b):
    new = video[video['Genre'] == a]
    channel = new.groupby('channelTitle')
    fin = channel[b].sum().sort_values(ascending=False).head()
    sns.barplot(y=fin.index, x=fin.values)
    plt.show()

widgets.interact(graph, a=c,b = conn)


### Saving the Dataframe to a CSV file for preprocessing and model building

In [ ]:
video.to_csv('grouped_data.csv')

## Final models: feature engineering + like-rate target (held-out R2 on raw likes)
RF 0.915, LightGBM 0.903, XGBoost 0.902, average 0.911.


In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
g = pd.read_csv('grouped_data.csv'); g['publishedAt'] = pd.to_datetime(g['publishedAt']); g['trending_date'] = pd.to_datetime(g['trending_date'])
d = pd.DataFrame(index=g.index); d['lv'] = np.log1p(g.view_count); d['lc'] = np.log1p(g.comment_count); d['ld'] = np.log1p(g.dislikes)
d['c_v'] = g.comment_count/(g.view_count+1); d['d_v'] = g.dislikes/(g.view_count+1); d['d_c'] = g.dislikes/(g.comment_count+1)
d['days'] = (g.trending_date-g.publishedAt).dt.days; d['pyear'] = g.p_year; d['hour'] = g.publishedAt.dt.hour; d['dow'] = g.publishedAt.dt.dayofweek
d['cd'] = g.comments_disabled.astype(int); d['rd'] = g.ratings_disabled.astype(int); d['cat'] = g.categoryId; d['dis_zero'] = (g.dislikes==0).astype(int)
d['ch_n'] = g.groupby('channelId').video_id.transform('count'); d['ch_lv'] = g.groupby('channelId').view_count.transform(lambda s: np.log1p(s).mean())
y = g.likes; v = g.view_count.values; r_all = np.log((y.values+1)/(v+1)); ch = g.channelId.values
Xtr, Xte, ytr, yte = train_test_split(d, y, test_size=0.25, random_state=3)
tr, te = Xtr.index.values, Xte.index.values; gm = r_all[tr].mean()
def enc(fi, ai): s = pd.DataFrame({'c': ch[fi], 'r': r_all[fi]}).groupby('c').r.agg(['sum','count']); a = pd.Series(ch[ai]); return ((a.map(s['sum']).fillna(0).values+gm*5)/(a.map(s['count']).fillna(0).values+5), a.map(s['count']).fillna(0).values)
oof = np.zeros(len(tr)); oofc = np.zeros(len(tr)); folds = list(KFold(5, shuffle=True, random_state=3).split(tr))
_ = [(oof.__setitem__(b, enc(tr[a], tr[b])[0]), oofc.__setitem__(b, enc(tr[a], tr[b])[1])) for a, b in folds]
te_e, te_c = enc(tr, te); Xtr2 = Xtr.assign(ch_rate=oof, ch_tr_n=oofc); Xte2 = Xte.assign(ch_rate=te_e, ch_tr_n=te_c)
models = {'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, max_features=0.5, n_jobs=-1, random_state=3), 'LightGBM': LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=3), 'XGBoost': XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=3)}
preds = {n: (v[te]+1)*np.exp(m.fit(Xtr2, r_all[tr]).predict(Xte2))-1 for n, m in models.items()}
preds['Average of 3'] = np.mean(list(preds.values()), axis=0)
out = pd.DataFrame({n: [r2_score(yte, p), r2_score(np.log1p(yte), np.log1p(np.clip(p, 0, None))), mean_absolute_error(yte, p)] for n, p in preds.items()}, index=['R2 on raw likes','R2 on log1p(likes)','MAE raw']).T
out.round(4)
